# Instalar Bibliotecas Necessárias

In [1]:
!pip install flair



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 9.0 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 68.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 125.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.2/16.2 MB 72.7 MB/s  0:00:006m0

In [1]:
!pip install textattack detoxify

#
# Note: No need to install textattack[tensorflow]. 
# If the current environment explicitly demands a text embedding framework that uses 
# TensorFlow Hub like Universal Sentence Encoder under the hood, it will prompt you at runtime. 

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 18.3 MB/s  0:00:00 eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 26.6 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━

# Importações

In [2]:
import numpy as np
print(np.__version__)

2.4.6


In [3]:
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import torch
import textattack
from textattack.datasets import HuggingFaceDataset
from textattack.models.wrappers import ModelWrapper


from textattack import AttackArgs
from textattack.datasets import Dataset
#from textattack import Attacker
from detoxify import Detoxify

import nltk
nltk.download('averaged_perceptron_tagger_eng') # TEM QUE TER ESSE ARQUIVO BAIXADO

textattack: Updating TextAttack package dependencies.
textattack: Downloading NLTK required packages.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package omw to /root/nltk_data...
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
/usr/local/lib/python3.13/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.13/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape s

True

# Modelo Vítima

# Exemplo utilizando o modelo da biblioteca [Detoxify](https://github.com/unitaryai/detoxify) para detecção de *hate speech*

Para este exemplo, utilizamos um modelo da biblioteca **Detoxify**, especializada em identificação de discurso de ódio (*hate speech*). A biblioteca disponibiliza três variantes de modelos baseados em arquiteturas *Transformer*:

## Modelos disponíveis

1. **`original`**  
   - *Baseado em*: **BERT** (bidirectional encoder representations)  
   - *Treinamento*: Fine-tuning com dados da **Wikipedia**  
   - *Aplicação*: Detecção geral de hate speech  

2. **`unbiased`**  
   - *Baseado em*: **BERT**  
   - *Dados de treino*: Comentários civis (*civil comments*)  
   - *Diferencial*: Projetado para **reduzir vieses** em moderação de conteúdo  

3. **`multilingual`**  
   - *Baseado em*: **XLM-RoBERTa** (cross-lingual model)  
   - *Dados*: Conjuntos multilingues (inglês + outros idiomas)  
   - *Vantagem*: Detecção em **múltiplos idiomas** com um único modelo  

In [4]:
#Exemplo de uso
model = Detoxify('unbiased')
results = model.predict(["I hate you"])

Downloading: "https://github.com/unitaryai/detoxify/releases/download/v0.3-alpha/toxic_debiased-c7548aa0.ckpt" to /root/.cache/torch/hub/checkpoints/toxic_debiased-c7548aa0.ckpt


100%|██████████| 476M/476M [00:07<00:00, 66.4MB/s] 
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


## Saída do modelo (Output Structure)
A predição retorna um dicionário Python com scores de probabilidade (0 a 1) para 7 classes de toxicidade:


In [6]:
results

{'toxicity': 0.8569894433021545,
 'severe_toxicity': 0.0005102233844809234,
 'obscene': 0.026927106082439423,
 'identity_attack': 0.043398309499025345,
 'insult': 0.26279398798942566,
 'threat': 0.03584827855229378,
 'sexual_explicit': 0.001184131600894034}

### **Wrapper**

Antes da realização do ataque a bibliteca textattack necessita que o modelo vítima esteja encapsulado em um wrapper para padronizar as entradas e saidas do modelo para que fique compatível com a implementação dos ataques da biblioteca.


In [7]:
class DetoxifyVictimModel(textattack.models.wrappers.ModelWrapper):
    def __init__(self, model_name='unbiased'):
        super().__init__()
        self.model = Detoxify(model_name)

    def __call__(self, texts):
        if isinstance(texts, str):
            texts = [texts]
        texts = [str(t) for t in texts]
        if len(texts) == 0:
            return torch.empty((0, 2), dtype=torch.float32)

        results = self.model.predict(texts)  # dict de listas
        return self.post_processing_detoxify_pred(results)

    def post_processing_detoxify_pred(self, results):
        keys = ['toxicity', 'severe_toxicity', 'insult', 'obscene',
                'sexual_explicit', 'identity_attack', 'threat']

        # Normaliza escalares -> listas, para o DataFrame sempre ter índice
        results = {k: (v if isinstance(v, (list, tuple)) else [v])
                   for k, v in results.items()}

        df_results = pd.DataFrame(results)
        pos = df_results[keys].max(axis=1)
        neg = 1 - pos
        return torch.tensor(
            pd.concat([neg, pos], axis=1).values, dtype=torch.float32
        )

# Dados
## [OLID](https://huggingface.co/datasets/christophsonntag/OLID) Dataset: Hierarchical Offensive Language Classification

**Descrição:**  
O *Offensive Language Identification Dataset (OLID)* é um corpus de **14,100 tweets** anotados via crowdsourcing, projetado para análise multinível de linguagem ofensiva em redes sociais. Desenvolvido para acompanhar o paper *"Predicting the Type and Target of Offensive Posts in Social Media"*, supera limitações de datasets anteriores ao classificar ofensas de forma hierárquica e contextualizada.

### Estrutura:
1. **Nível A (Task A)**  
   - Classificação binária: `OFF` (ofensivo) vs `NOT` (não-ofensivo)  

2. **Nível B (Task B)**  
   - Subtipos de ofensa:  
     - `TIN` (*Targeted Insult*): Ofensa direcionada  
     - `UNT` (*Untargeted*): Ofensa genérica (ex.: xingamentos sem alvo)  

3. **Nível C (Task C)**  
   - Identificação do alvo:  
     - `IND` (Indivíduo)  
     - `GRP` (Grupo - raça, gênero, etc.)  
     - `OTH` (Outro - entidades, organizações)  


In [8]:
splits = {'train': 'train.csv', 'test': 'test.csv'}
df = pd.read_csv("hf://datasets/christophsonntag/OLID/" + splits["train"])

In [9]:
df = df[['cleaned_tweet', 'subtask_a']].rename(columns={'cleaned_tweet':'text', 'subtask_a':'label'})
df.label = df.label.apply(lambda x: 1 if x == 'OFF' else 0)

In [10]:
df

,text,label
0,ask native americans take,1
1,go home drunk maga trump 2020,1
2,amazon investigating chinese employees selling...,0
3,someone vetaken piece shit volcano,1
4,obama wanted liberals illegals move red states,0
...,...,...
13235,sometimes get strong vibes people man vibe ten...,1
13236,benidorm ✅ creamfields ✅ maga ✅ shabby summer,0
13237,report garbage give crap,1
13238,pussy,1


In [11]:
#Neste caso a estrutura dos dados para a utilizaçao do textattack são uma lista com tuplas no seguinte formato: [('texto','classe')]
data = [(text, target) for text, target in zip(df['text'], df['label'])]
dataset = Dataset(data)

dataset[0:5]

[(OrderedDict([('text', 'ask native americans take')]), 1),
 (OrderedDict([('text', 'go home drunk maga trump 2020')]), 1),
 (OrderedDict([('text',
                'amazon investigating chinese employees selling internal data third party sellers looking edge competitive marketplace amazon maga kag china tcot')]),
  0),
 (OrderedDict([('text', 'someone vetaken piece shit volcano')]), 1),
 (OrderedDict([('text', 'obama wanted liberals illegals move red states')]),
  0)]

# Ataque GNLA


# [GNLA](https://arxiv.org/abs/2012.14956): Ataque Adversarial Baseado em Algoritmos Genéticos

## Visão Geral
O **GNLA** é uma técnica de ataque adversarial *black-box* que utiliza algoritmos genéticos para encontrar perturbações eficazes. Diferente de métodos baseados em gradiente, esta abordagem otimiza iterativamente substituições de palavras através de um processo evolutivo, mantendo restrições de similaridade semântica enquanto força erros de classificação.

## Mecanismo do Ataque
Processo em 4 fases:

1. **Perturbação Inicial**  
   - Substitui até 30% das palavras por sinônimos (top 50)  
   - Mede impacto na classificação e similaridade semântica

2. **Redução do Espaço de Busca**  
   - Identifica palavras-chave que mais afetam a predição  
   - Elimina modificações irrelevantes via teste de reversão

3. **Otimização Genética**  
   - População: Candidatos com combinações de substituições  
   - Fitness: Balanceia:  
     * Mudança na classe alvo (maximizar)  
     * Similaridade semântica (preservar)

4. **Seleção Final**  
   - Retorna a perturbação que engana o modelo  
   - Garante legibilidade humana (score semântico > threshold)

## Redefinição de Search Function e Classe Geral de Ataque para utilização de ataque customizado

In [12]:
# @title
"""
Search Method Abstract Class
===============================
"""


from abc import ABC, abstractmethod

from textattack.shared.utils import default_class_repr


class SearchMethod(ABC):
    """This is an abstract class that contains main helper functionality for
    search methods.

    A search method is a strategy for applying transformations until the
    goal is met or the search is exhausted.
    """

    def __call__(self, initial_result):
        """Ensures access to necessary functions, then calls
        ``_perform_search``"""
        if not hasattr(self, "get_transformations"):
            raise AttributeError(
                "Search Method must have access to get_transformations method"
            )
        if not hasattr(self, "get_goal_results"):
            raise AttributeError(
                "Search Method must have access to get_goal_results method"
            )
        if not hasattr(self, "filter_transformations"):
            raise AttributeError(
                "Search Method must have access to filter_transformations method"
            )

        if not self.is_black_box and not hasattr(self, "get_model"):
            raise AttributeError(
                "Search Method must have access to get_model method if it is a white-box method"
            )
        result = self._perform_search(initial_result)
        # ensure that the number of queries for this GoalFunctionResult is up-to-date
        result.num_queries = self.goal_function.num_queries
        return result

    @abstractmethod
    def _perform_search(self, initial_result):
        """Perturbs `attacked_text` from ``initial_result`` until goal is
        reached or search is exhausted.

        Must be overridden by specific search methods.
        """
        raise NotImplementedError()

    def check_transformation_compatibility(self, transformation):
        """Determines whether this search method is compatible with
        ``transformation``."""
        return True

    @property
    def is_black_box(self):
        """Returns `True` if search method does not require access to victim
        model's internal states."""
        raise NotImplementedError()

    def extra_repr_keys(self):
        return []

    __repr__ = __str__ = default_class_repr

In [13]:
# @title
"""
Attack Class
============
"""

from collections import OrderedDict
from typing import List, Union

import lru
import torch

import textattack
from textattack.attack_results import (
    FailedAttackResult,
    MaximizedAttackResult,
    SkippedAttackResult,
    SuccessfulAttackResult,
)
from textattack.constraints import Constraint, PreTransformationConstraint
from textattack.goal_function_results import GoalFunctionResultStatus
from textattack.goal_functions import GoalFunction
from textattack.models.wrappers import ModelWrapper
#from textattack.search_methods import SearchMethod
from textattack.shared import AttackedText, utils
from textattack.transformations import CompositeTransformation, Transformation


class Attack:
    """An attack generates adversarial examples on text.

    An attack is comprised of a goal function, constraints, transformation, and a search method. Use :meth:`attack` method to attack one sample at a time.

    Args:
        goal_function (:class:`~textattack.goal_functions.GoalFunction`):
            A function for determining how well a perturbation is doing at achieving the attack's goal.
        constraints (list of :class:`~textattack.constraints.Constraint` or :class:`~textattack.constraints.PreTransformationConstraint`):
            A list of constraints to add to the attack, defining which perturbations are valid.
        transformation (:class:`~textattack.transformations.Transformation`):
            The transformation applied at each step of the attack.
        search_method (:class:`~textattack.search_methods.SearchMethod`):
            The method for exploring the search space of possible perturbations
        transformation_cache_size (:obj:`int`, `optional`, defaults to :obj:`2**15`):
            The number of items to keep in the transformations cache
        constraint_cache_size (:obj:`int`, `optional`, defaults to :obj:`2**15`):
            The number of items to keep in the constraints cache

    Example::

        >>> import textattack
        >>> import transformers

        >>> # Load model, tokenizer, and model_wrapper
        >>> model = transformers.AutoModelForSequenceClassification.from_pretrained("textattack/bert-base-uncased-imdb")
        >>> tokenizer = transformers.AutoTokenizer.from_pretrained("textattack/bert-base-uncased-imdb")
        >>> model_wrapper = textattack.models.wrappers.HuggingFaceModelWrapper(model, tokenizer)

        >>> # Construct our four components for `Attack`
        >>> from textattack.constraints.pre_transformation import RepeatModification, StopwordModification
        >>> from textattack.constraints.semantics import WordEmbeddingDistance
        >>> from textattack.transformations import WordSwapEmbedding
        >>> from textattack.search_methods import GreedyWordSwapWIR

        >>> goal_function = textattack.goal_functions.UntargetedClassification(model_wrapper)
        >>> constraints = [
        ...     RepeatModification(),
        ...     StopwordModification(),
        ...     WordEmbeddingDistance(min_cos_sim=0.9)
        ... ]
        >>> transformation = WordSwapEmbedding(max_candidates=50)
        >>> search_method = GreedyWordSwapWIR(wir_method="delete")

        >>> # Construct the actual attack
        >>> attack = textattack.Attack(goal_function, constraints, transformation, search_method)

        >>> input_text = "I really enjoyed the new movie that came out last month."
        >>> label = 1 #Positive
        >>> attack_result = attack.attack(input_text, label)
    """

    def __init__(
        self,
        goal_function: GoalFunction,
        constraints: List[Union[Constraint, PreTransformationConstraint]],
        transformation: Transformation,
        search_method: SearchMethod,
        transformation_cache_size=2**15,
        constraint_cache_size=2**15,
    ):
        """Initialize an attack object.

        Attacks can be run multiple times.
        """
        assert isinstance(
            goal_function, GoalFunction
        ), f"`goal_function` must be of type `textattack.goal_functions.GoalFunction`, but got type `{type(goal_function)}`."
        assert isinstance(
            constraints, list
        ), "`constraints` must be a list of `textattack.constraints.Constraint` or `textattack.constraints.PreTransformationConstraint`."
        for c in constraints:
            assert isinstance(
                c, (Constraint, PreTransformationConstraint)
            ), "`constraints` must be a list of `textattack.constraints.Constraint` or `textattack.constraints.PreTransformationConstraint`."
        assert isinstance(
            transformation, Transformation
        ), f"`transformation` must be of type `textattack.transformations.Transformation`, but got type `{type(transformation)}`."
#        assert isinstance(
#            search_method, SearchMethod
#        ), f"`search_method` must be of type `textattack.search_methods.SearchMethod`, but got type `{type(search_method)}`."

        self.goal_function = goal_function
        self.search_method = search_method
        self.transformation = transformation
        self.is_black_box = (
            getattr(transformation, "is_black_box", True) and search_method.is_black_box
        )

        if not self.search_method.check_transformation_compatibility(
            self.transformation
        ):
            raise ValueError(
                f"SearchMethod {self.search_method} incompatible with transformation {self.transformation}"
            )

        self.constraints = []
        self.pre_transformation_constraints = []
        for constraint in constraints:
            if isinstance(
                constraint,
                textattack.constraints.PreTransformationConstraint,
            ):
                self.pre_transformation_constraints.append(constraint)
            else:
                self.constraints.append(constraint)

        # Check if we can use transformation cache for our transformation.
        if not self.transformation.deterministic:
            self.use_transformation_cache = False
        elif isinstance(self.transformation, CompositeTransformation):
            self.use_transformation_cache = True
            for t in self.transformation.transformations:
                if not t.deterministic:
                    self.use_transformation_cache = False
                    break
        else:
            self.use_transformation_cache = True
        self.transformation_cache_size = transformation_cache_size
        self.transformation_cache = lru.LRU(transformation_cache_size)

        self.constraint_cache_size = constraint_cache_size
        self.constraints_cache = lru.LRU(constraint_cache_size)

        # Give search method access to functions for getting transformations and evaluating them
        self.search_method.get_transformations = self.get_transformations
        # Give search method access to self.goal_function for model query count, etc.
        self.search_method.goal_function = self.goal_function
        # The search method only needs access to the first argument. The second is only used
        # by the attack class when checking whether to skip the sample
        self.search_method.get_goal_results = self.goal_function.get_results

        # Give search method access to get indices which need to be ordered / searched
        self.search_method.get_indices_to_order = self.get_indices_to_order

        self.search_method.filter_transformations = self.filter_transformations

    def clear_cache(self, recursive=True):
        self.constraints_cache.clear()
        if self.use_transformation_cache:
            self.transformation_cache.clear()
        if recursive:
            self.goal_function.clear_cache()
            for constraint in self.constraints:
                if hasattr(constraint, "clear_cache"):
                    constraint.clear_cache()

    def cpu_(self):
        """Move any `torch.nn.Module` models that are part of Attack to CPU."""
        visited = set()

        def to_cpu(obj):
            visited.add(id(obj))
            if isinstance(obj, torch.nn.Module):
                obj.cpu()
            elif isinstance(
                obj,
                (
                    Attack,
                    GoalFunction,
                    Transformation,
                    SearchMethod,
                    Constraint,
                    PreTransformationConstraint,
                    ModelWrapper,
                ),
            ):
                for key in obj.__dict__:
                    s_obj = obj.__dict__[key]
                    if id(s_obj) not in visited:
                        to_cpu(s_obj)
            elif isinstance(obj, (list, tuple)):
                for item in obj:
                    if id(item) not in visited and isinstance(
                        item, (Transformation, Constraint, PreTransformationConstraint)
                    ):
                        to_cpu(item)

        to_cpu(self)

    def cuda_(self):
        """Move any `torch.nn.Module` models that are part of Attack to GPU."""
        visited = set()

        def to_cuda(obj):
            visited.add(id(obj))
            if isinstance(obj, torch.nn.Module):
                obj.to(textattack.shared.utils.device)
            elif isinstance(
                obj,
                (
                    Attack,
                    GoalFunction,
                    Transformation,
                    SearchMethod,
                    Constraint,
                    PreTransformationConstraint,
                    ModelWrapper,
                ),
            ):
                for key in obj.__dict__:
                    s_obj = obj.__dict__[key]
                    if id(s_obj) not in visited:
                        to_cuda(s_obj)
            elif isinstance(obj, (list, tuple)):
                for item in obj:
                    if id(item) not in visited and isinstance(
                        item, (Transformation, Constraint, PreTransformationConstraint)
                    ):
                        to_cuda(item)

        to_cuda(self)

    def get_indices_to_order(self, current_text, **kwargs):
        """Applies ``pre_transformation_constraints`` to ``text`` to get all
        the indices that can be used to search and order.

        Args:
            current_text: The current ``AttackedText`` for which we need to find indices are eligible to be ordered.
        Returns:
            The length and the filtered list of indices which search methods can use to search/order.
        """

        indices_to_order = self.transformation(
            current_text,
            pre_transformation_constraints=self.pre_transformation_constraints,
            return_indices=True,
            **kwargs,
        )

        len_text = len(indices_to_order)

        # Convert indices_to_order to list for easier shuffling later
        return len_text, list(indices_to_order)

    def _get_transformations_uncached(self, current_text, original_text=None, **kwargs):
        """Applies ``self.transformation`` to ``text``, then filters the list
        of possible transformations through the applicable constraints.

        Args:
            current_text: The current ``AttackedText`` on which to perform the transformations.
            original_text: The original ``AttackedText`` from which the attack started.
        Returns:
            A filtered list of transformations where each transformation matches the constraints
        """
        transformed_texts = self.transformation(
            current_text,
            pre_transformation_constraints=self.pre_transformation_constraints,
            **kwargs,
        )

        return transformed_texts

    def get_transformations(self, current_text, original_text=None, **kwargs):
        """Applies ``self.transformation`` to ``text``, then filters the list
        of possible transformations through the applicable constraints.

        Args:
            current_text: The current ``AttackedText`` on which to perform the transformations.
            original_text: The original ``AttackedText`` from which the attack started.
        Returns:
            A filtered list of transformations where each transformation matches the constraints
        """
        if not self.transformation:
            raise RuntimeError(
                "Cannot call `get_transformations` without a transformation."
            )

        if self.use_transformation_cache:
            cache_key = tuple([current_text] + sorted(kwargs.items()))
            if utils.hashable(cache_key) and cache_key in self.transformation_cache:
                # promote transformed_text to the top of the LRU cache
                self.transformation_cache[cache_key] = self.transformation_cache[
                    cache_key
                ]
                transformed_texts = list(self.transformation_cache[cache_key])
            else:
                transformed_texts = self._get_transformations_uncached(
                    current_text, original_text, **kwargs
                )
                if utils.hashable(cache_key):
                    self.transformation_cache[cache_key] = tuple(transformed_texts)
        else:
            transformed_texts = self._get_transformations_uncached(
                current_text, original_text, **kwargs
            )

        return self.filter_transformations(
            transformed_texts, current_text, original_text
        )

    def _filter_transformations_uncached(
        self, transformed_texts, current_text, original_text=None
    ):
        """Filters a list of potential transformed texts based on
        ``self.constraints``

        Args:
            transformed_texts: A list of candidate transformed ``AttackedText`` to filter.
            current_text: The current ``AttackedText`` on which the transformation was applied.
            original_text: The original ``AttackedText`` from which the attack started.
        """
        filtered_texts = transformed_texts[:]
        for C in self.constraints:
            if len(filtered_texts) == 0:
                break
            if C.compare_against_original:
                if not original_text:
                    raise ValueError(
                        f"Missing `original_text` argument when constraint {type(C)} is set to compare against `original_text`"
                    )

                filtered_texts = C.call_many(filtered_texts, original_text)
            else:
                filtered_texts = C.call_many(filtered_texts, current_text)
        # Default to false for all original transformations.
        for original_transformed_text in transformed_texts:
            self.constraints_cache[(current_text, original_transformed_text)] = False
        # Set unfiltered transformations to True in the cache.
        for filtered_text in filtered_texts:
            self.constraints_cache[(current_text, filtered_text)] = True
        return filtered_texts

    def filter_transformations(
        self, transformed_texts, current_text, original_text=None
    ):
        """Filters a list of potential transformed texts based on
        ``self.constraints`` Utilizes an LRU cache to attempt to avoid
        recomputing common transformations.

        Args:
            transformed_texts: A list of candidate transformed ``AttackedText`` to filter.
            current_text: The current ``AttackedText`` on which the transformation was applied.
            original_text: The original ``AttackedText`` from which the attack started.
        """
        # Remove any occurences of current_text in transformed_texts
        transformed_texts = [
            t for t in transformed_texts if t.text != current_text.text
        ]
        # Populate cache with transformed_texts
        uncached_texts = []
        filtered_texts = []
        for transformed_text in transformed_texts:
            if (current_text, transformed_text) not in self.constraints_cache:
                uncached_texts.append(transformed_text)
            else:
                # promote transformed_text to the top of the LRU cache
                self.constraints_cache[(current_text, transformed_text)] = (
                    self.constraints_cache[(current_text, transformed_text)]
                )
                if self.constraints_cache[(current_text, transformed_text)]:
                    filtered_texts.append(transformed_text)
        filtered_texts += self._filter_transformations_uncached(
            uncached_texts, current_text, original_text=original_text
        )
        # Sort transformations to ensure order is preserved between runs
        filtered_texts.sort(key=lambda t: t.text)
        return filtered_texts

    def _attack(self, initial_result):
        """Calls the ``SearchMethod`` to perturb the ``AttackedText`` stored in
        ``initial_result``.

        Args:
            initial_result: The initial ``GoalFunctionResult`` from which to perturb.

        Returns:
            A ``SuccessfulAttackResult``, ``FailedAttackResult``,
                or ``MaximizedAttackResult``.
        """
        final_result = self.search_method(initial_result)
        self.clear_cache()
        if final_result.goal_status == GoalFunctionResultStatus.SUCCEEDED:
            result = SuccessfulAttackResult(
                initial_result,
                final_result,
            )
        elif final_result.goal_status == GoalFunctionResultStatus.SEARCHING:
            result = FailedAttackResult(
                initial_result,
                final_result,
            )
        elif final_result.goal_status == GoalFunctionResultStatus.MAXIMIZING:
            result = MaximizedAttackResult(
                initial_result,
                final_result,
            )
        else:
            raise ValueError(f"Unrecognized goal status {final_result.goal_status}")
        return result

    def attack(self, example, ground_truth_output):
        """Attack a single example.

        Args:
            example (:obj:`str`, :obj:`OrderedDict[str, str]` or :class:`~textattack.shared.AttackedText`):
                Example to attack. It can be a single string or an `OrderedDict` where
                keys represent the input fields (e.g. "premise", "hypothesis") and the values are the actual input textx.
                Also accepts :class:`~textattack.shared.AttackedText` that wraps around the input.
            ground_truth_output(:obj:`int`, :obj:`float` or :obj:`str`):
                Ground truth output of `example`.
                For classification tasks, it should be an integer representing the ground truth label.
                For regression tasks (e.g. STS), it should be the target value.
                For seq2seq tasks (e.g. translation), it should be the target string.
        Returns:
            :class:`~textattack.attack_results.AttackResult` that represents the result of the attack.
        """
        assert isinstance(
            example, (str, OrderedDict, AttackedText)
        ), "`example` must either be `str`, `collections.OrderedDict`, `textattack.shared.AttackedText`."
        if isinstance(example, (str, OrderedDict)):
            example = AttackedText(example)

        assert isinstance(
            ground_truth_output, (int, str)
        ), "`ground_truth_output` must either be `str` or `int`."
        goal_function_result, _ = self.goal_function.init_attack_example(
            example, ground_truth_output
        )
        if goal_function_result.goal_status == GoalFunctionResultStatus.SKIPPED:
            return SkippedAttackResult(goal_function_result)
        else:
            result = self._attack(goal_function_result)
            return result

    def __repr__(self):
        """Prints attack parameters in a human-readable string.

        Inspired by the readability of printing PyTorch nn.Modules:
        https://github.com/pytorch/pytorch/blob/master/torch/nn/modules/module.py
        """
        main_str = "Attack" + "("
        lines = []

        lines.append(utils.add_indent(f"(search_method): {self.search_method}", 2))
        # self.goal_function
        lines.append(utils.add_indent(f"(goal_function):  {self.goal_function}", 2))
        # self.transformation
        lines.append(utils.add_indent(f"(transformation):  {self.transformation}", 2))
        # self.constraints
        constraints_lines = []
        constraints = self.constraints + self.pre_transformation_constraints
        if len(constraints):
            for i, constraint in enumerate(constraints):
                constraints_lines.append(utils.add_indent(f"({i}): {constraint}", 2))
            constraints_str = utils.add_indent("\n" + "\n".join(constraints_lines), 2)
        else:
            constraints_str = "None"
        lines.append(utils.add_indent(f"(constraints): {constraints_str}", 2))
        # self.is_black_box
        lines.append(utils.add_indent(f"(is_black_box):  {self.is_black_box}", 2))
        main_str += "\n  " + "\n  ".join(lines) + "\n"
        main_str += ")"
        return main_str

    def __getstate__(self):
        state = self.__dict__.copy()
        state["transformation_cache"] = None
        state["constraints_cache"] = None
        return state

    def __setstate__(self, state):
        self.__dict__ = state
        self.transformation_cache = lru.LRU(self.transformation_cache_size)
        self.constraints_cache = lru.LRU(self.constraint_cache_size)

    __str__ = __repr__


In [14]:
# @title
"""
Attacker Class
==============
"""

import collections
import logging
import multiprocessing as mp
import os
import queue
import random
import traceback

import torch
import tqdm

import textattack
from textattack.attack_results import (
    FailedAttackResult,
    MaximizedAttackResult,
    SkippedAttackResult,
    SuccessfulAttackResult,
)
from textattack.shared.utils import logger

from textattack.attack_args import AttackArgs


class Attacker:
    """Class for running attacks on a dataset with specified parameters. This
    class uses the :class:`~textattack.Attack` to actually run the attacks,
    while also providing useful features such as parallel processing,
    saving/resuming from a checkpint, logging to files and stdout.

    Args:
        attack (:class:`~textattack.Attack`):
            :class:`~textattack.Attack` used to actually carry out the attack.
        dataset (:class:`~textattack.datasets.Dataset`):
            Dataset to attack.
        attack_args (:class:`~textattack.AttackArgs`):
            Arguments for attacking the dataset. For default settings, look at the `AttackArgs` class.

    Example::

        >>> import textattack
        >>> import transformers

        >>> model = transformers.AutoModelForSequenceClassification.from_pretrained("textattack/bert-base-uncased-imdb")
        >>> tokenizer = transformers.AutoTokenizer.from_pretrained("textattack/bert-base-uncased-imdb")
        >>> model_wrapper = textattack.models.wrappers.HuggingFaceModelWrapper(model, tokenizer)

        >>> attack = textattack.attack_recipes.TextFoolerJin2019.build(model_wrapper)
        >>> dataset = textattack.datasets.HuggingFaceDataset("imdb", split="test")

        >>> # Attack 20 samples with CSV logging and checkpoint saved every 5 interval
        >>> attack_args = textattack.AttackArgs(
        ...     num_examples=20,
        ...     log_to_csv="log.csv",
        ...     checkpoint_interval=5,
        ...     checkpoint_dir="checkpoints",
        ...     disable_stdout=True
        ... )

        >>> attacker = textattack.Attacker(attack, dataset, attack_args)
        >>> attacker.attack_dataset()
    """

    def __init__(self, attack, dataset, attack_args=None):
        assert isinstance(
            attack, Attack
        ), f"`attack` argument must be of type `textattack.Attack`, but got type of `{type(attack)}`."
        assert isinstance(
            dataset, textattack.datasets.Dataset
        ), f"`dataset` must be of type `textattack.datasets.Dataset`, but got type `{type(dataset)}`."

        if attack_args:
            assert isinstance(
                attack_args, AttackArgs
            ), f"`attack_args` must be of type `textattack.AttackArgs`, but got type `{type(attack_args)}`."
        else:
            attack_args = AttackArgs()

        self.attack = attack
        self.dataset = dataset
        self.attack_args = attack_args
        self.attack_log_manager = None

        # This is to be set if loading from a checkpoint
        self._checkpoint = None

    def _get_worklist(self, start, end, num_examples, shuffle):
        if end - start < num_examples:
            logger.warn(
                f"Attempting to attack {num_examples} samples when only {end-start} are available."
            )
        candidates = list(range(start, end))
        if shuffle:
            random.shuffle(candidates)
        worklist = collections.deque(candidates[:num_examples])
        candidates = collections.deque(candidates[num_examples:])
        assert (len(worklist) + len(candidates)) == (end - start)
        return worklist, candidates

    def _attack(self):
        """Internal method that carries out attack.

        No parallel processing is involved.
        """
        if torch.cuda.is_available():
            self.attack.cuda_()

        if self._checkpoint:
            num_remaining_attacks = self._checkpoint.num_remaining_attacks
            worklist = self._checkpoint.worklist
            worklist_candidates = self._checkpoint.worklist_candidates
            logger.info(
                f"Recovered from checkpoint previously saved at {self._checkpoint.datetime}."
            )
        else:
            if self.attack_args.num_successful_examples:
                num_remaining_attacks = self.attack_args.num_successful_examples
                # We make `worklist` deque (linked-list) for easy pop and append.
                # Candidates are other samples we can attack if we need more samples.
                worklist, worklist_candidates = self._get_worklist(
                    self.attack_args.num_examples_offset,
                    len(self.dataset),
                    self.attack_args.num_successful_examples,
                    self.attack_args.shuffle,
                )
            else:
                num_remaining_attacks = self.attack_args.num_examples
                # We make `worklist` deque (linked-list) for easy pop and append.
                # Candidates are other samples we can attack if we need more samples.
                worklist, worklist_candidates = self._get_worklist(
                    self.attack_args.num_examples_offset,
                    len(self.dataset),
                    self.attack_args.num_examples,
                    self.attack_args.shuffle,
                )

        if not self.attack_args.silent:
            print(self.attack, "\n")

        pbar = tqdm.tqdm(total=num_remaining_attacks, smoothing=0, dynamic_ncols=True)
        if self._checkpoint:
            num_results = self._checkpoint.results_count
            num_failures = self._checkpoint.num_failed_attacks
            num_skipped = self._checkpoint.num_skipped_attacks
            num_successes = self._checkpoint.num_successful_attacks
        else:
            num_results = 0
            num_failures = 0
            num_skipped = 0
            num_successes = 0

        sample_exhaustion_warned = False
        while worklist:
            idx = worklist.popleft()
            try:
                example, ground_truth_output = self.dataset[idx]
            except IndexError:
                continue
            example = textattack.shared.AttackedText(example)
            if self.dataset.label_names is not None:
                example.attack_attrs["label_names"] = self.dataset.label_names
            try:
                result = self.attack.attack(example, ground_truth_output)
            except Exception as e:
                raise e
            if (
                isinstance(result, SkippedAttackResult) and self.attack_args.attack_n
            ) or (
                not isinstance(result, SuccessfulAttackResult)
                and self.attack_args.num_successful_examples
            ):
                if worklist_candidates:
                    next_sample = worklist_candidates.popleft()
                    worklist.append(next_sample)
                else:
                    if not sample_exhaustion_warned:
                        logger.warn("Ran out of samples to attack!")
                        sample_exhaustion_warned = True
            else:
                pbar.update(1)

            self.attack_log_manager.log_result(result)
            if not self.attack_args.disable_stdout and not self.attack_args.silent:
                print("\n")
            num_results += 1

            if isinstance(result, SkippedAttackResult):
                num_skipped += 1
            if isinstance(result, (SuccessfulAttackResult, MaximizedAttackResult)):
                num_successes += 1
            if isinstance(result, FailedAttackResult):
                num_failures += 1
            pbar.set_description(
                f"[Succeeded / Failed / Skipped / Total] {num_successes} / {num_failures} / {num_skipped} / {num_results}"
            )

            if (
                self.attack_args.checkpoint_interval
                and len(self.attack_log_manager.results)
                % self.attack_args.checkpoint_interval
                == 0
            ):
                new_checkpoint = textattack.shared.AttackCheckpoint(
                    self.attack_args,
                    self.attack_log_manager,
                    worklist,
                    worklist_candidates,
                )
                new_checkpoint.save()
                self.attack_log_manager.flush()

        pbar.close()
        print()
        # Enable summary stdout
        if not self.attack_args.silent and self.attack_args.disable_stdout:
            self.attack_log_manager.enable_stdout()

        if self.attack_args.enable_advance_metrics:
            self.attack_log_manager.enable_advance_metrics = True

        self.attack_log_manager.log_summary()
        self.attack_log_manager.flush()
        print()

    def _attack_parallel(self):
        pytorch_multiprocessing_workaround()

        if self._checkpoint:
            num_remaining_attacks = self._checkpoint.num_remaining_attacks
            worklist = self._checkpoint.worklist
            worklist_candidates = self._checkpoint.worklist_candidates
            logger.info(
                f"Recovered from checkpoint previously saved at {self._checkpoint.datetime}."
            )
        else:
            if self.attack_args.num_successful_examples:
                num_remaining_attacks = self.attack_args.num_successful_examples
                # We make `worklist` deque (linked-list) for easy pop and append.
                # Candidates are other samples we can attack if we need more samples.
                worklist, worklist_candidates = self._get_worklist(
                    self.attack_args.num_examples_offset,
                    len(self.dataset),
                    self.attack_args.num_successful_examples,
                    self.attack_args.shuffle,
                )
            else:
                num_remaining_attacks = self.attack_args.num_examples
                # We make `worklist` deque (linked-list) for easy pop and append.
                # Candidates are other samples we can attack if we need more samples.
                worklist, worklist_candidates = self._get_worklist(
                    self.attack_args.num_examples_offset,
                    len(self.dataset),
                    self.attack_args.num_examples,
                    self.attack_args.shuffle,
                )

        in_queue = torch.multiprocessing.Queue()
        out_queue = torch.multiprocessing.Queue()
        for i in worklist:
            try:
                example, ground_truth_output = self.dataset[i]
                example = textattack.shared.AttackedText(example)
                if self.dataset.label_names is not None:
                    example.attack_attrs["label_names"] = self.dataset.label_names
                in_queue.put((i, example, ground_truth_output))
            except IndexError:
                raise IndexError(
                    f"Tried to access element at {i} in dataset of size {len(self.dataset)}."
                )

        # We reserve the first GPU for coordinating workers.
        num_gpus = torch.cuda.device_count()
        num_workers = self.attack_args.num_workers_per_device * num_gpus
        logger.info(f"Running {num_workers} worker(s) on {num_gpus} GPU(s).")

        # Lock for synchronization
        lock = mp.Lock()

        # We move Attacker (and its components) to CPU b/c we don't want models using wrong GPU in worker processes.
        self.attack.cpu_()
        torch.cuda.empty_cache()

        # Start workers.
        worker_pool = torch.multiprocessing.Pool(
            num_workers,
            attack_from_queue,
            (
                self.attack,
                self.attack_args,
                num_gpus,
                mp.Value("i", 1, lock=False),
                lock,
                in_queue,
                out_queue,
            ),
        )

        # Log results asynchronously and update progress bar.
        if self._checkpoint:
            num_results = self._checkpoint.results_count
            num_failures = self._checkpoint.num_failed_attacks
            num_skipped = self._checkpoint.num_skipped_attacks
            num_successes = self._checkpoint.num_successful_attacks
        else:
            num_results = 0
            num_failures = 0
            num_skipped = 0
            num_successes = 0

        logger.info(f"Worklist size: {len(worklist)}")
        logger.info(f"Worklist candidate size: {len(worklist_candidates)}")

        sample_exhaustion_warned = False
        pbar = tqdm.tqdm(total=num_remaining_attacks, smoothing=0, dynamic_ncols=True)
        while worklist:
            idx, result = out_queue.get(block=True)
            worklist.remove(idx)

            if isinstance(result, tuple) and isinstance(result[0], Exception):
                logger.error(
                    f'Exception encountered for input "{self.dataset[idx][0]}".'
                )
                error_trace = result[1]
                logger.error(error_trace)
                in_queue.close()
                in_queue.join_thread()
                out_queue.close()
                out_queue.join_thread()
                worker_pool.terminate()
                worker_pool.join()
                return
            elif (
                isinstance(result, SkippedAttackResult) and self.attack_args.attack_n
            ) or (
                not isinstance(result, SuccessfulAttackResult)
                and self.attack_args.num_successful_examples
            ):
                if worklist_candidates:
                    next_sample = worklist_candidates.popleft()
                    example, ground_truth_output = self.dataset[next_sample]
                    example = textattack.shared.AttackedText(example)
                    if self.dataset.label_names is not None:
                        example.attack_attrs["label_names"] = self.dataset.label_names
                    worklist.append(next_sample)
                    in_queue.put((next_sample, example, ground_truth_output))
                else:
                    if not sample_exhaustion_warned:
                        logger.warn("Ran out of samples to attack!")
                        sample_exhaustion_warned = True
            else:
                pbar.update()

            self.attack_log_manager.log_result(result)
            num_results += 1

            if isinstance(result, SkippedAttackResult):
                num_skipped += 1
            if isinstance(result, (SuccessfulAttackResult, MaximizedAttackResult)):
                num_successes += 1
            if isinstance(result, FailedAttackResult):
                num_failures += 1
            pbar.set_description(
                f"[Succeeded / Failed / Skipped / Total] {num_successes} / {num_failures} / {num_skipped} / {num_results}"
            )

            if (
                self.attack_args.checkpoint_interval
                and len(self.attack_log_manager.results)
                % self.attack_args.checkpoint_interval
                == 0
            ):
                new_checkpoint = textattack.shared.AttackCheckpoint(
                    self.attack_args,
                    self.attack_log_manager,
                    worklist,
                    worklist_candidates,
                )
                new_checkpoint.save()
                self.attack_log_manager.flush()

        # Send sentinel values to worker processes
        for _ in range(num_workers):
            in_queue.put(("END", "END", "END"))
        worker_pool.close()
        worker_pool.join()

        pbar.close()
        print()
        # Enable summary stdout.
        if not self.attack_args.silent and self.attack_args.disable_stdout:
            self.attack_log_manager.enable_stdout()

        if self.attack_args.enable_advance_metrics:
            self.attack_log_manager.enable_advance_metrics = True

        self.attack_log_manager.log_summary()
        self.attack_log_manager.flush()
        print()

    def attack_dataset(self):
        """Attack the dataset.

        Returns:
            :obj:`list[AttackResult]` - List of :class:`~textattack.attack_results.AttackResult` obtained after attacking the given dataset..
        """
        if self.attack_args.silent:
            logger.setLevel(logging.ERROR)

        if self.attack_args.query_budget:
            self.attack.goal_function.query_budget = self.attack_args.query_budget

        if not self.attack_log_manager:
            self.attack_log_manager = AttackArgs.create_loggers_from_args(
                self.attack_args
            )

        textattack.shared.utils.set_seed(self.attack_args.random_seed)
        if self.dataset.shuffled and self.attack_args.checkpoint_interval:
            # Not allowed b/c we cannot recover order of shuffled data
            raise ValueError(
                "Cannot use `--checkpoint-interval` with dataset that has been internally shuffled."
            )

        self.attack_args.num_examples = (
            len(self.dataset)
            if self.attack_args.num_examples == -1
            else self.attack_args.num_examples
        )
        if self.attack_args.parallel:
            if torch.cuda.device_count() == 0:
                raise Exception(
                    "Found no GPU on your system. To run attacks in parallel, GPU is required."
                )
            self._attack_parallel()
        else:
            self._attack()

        if self.attack_args.silent:
            logger.setLevel(logging.INFO)

        return self.attack_log_manager.results

    def update_attack_args(self, **kwargs):
        """To update any attack args, pass the new argument as keyword argument
        to this function.

        Examples::

        >>> attacker = #some instance of Attacker
        >>> # To switch to parallel mode and increase checkpoint interval from 100 to 500
        >>> attacker.update_attack_args(parallel=True, checkpoint_interval=500)
        """
        for k in kwargs:
            if hasattr(self.attack_args, k):
                self.attack_args.k = kwargs[k]
            else:
                raise ValueError(f"`textattack.AttackArgs` does not have field {k}.")

    @classmethod
    def from_checkpoint(cls, attack, dataset, checkpoint):
        """Resume attacking from a saved checkpoint. Attacker and dataset must
        be recovered by the user again, while attack args are loaded from the
        saved checkpoint.

        Args:
            attack (:class:`~textattack.Attack`):
                Attack object for carrying out the attack.
            dataset (:class:`~textattack.datasets.Dataset`):
                Dataset to attack.
            checkpoint (:obj:`Union[str, :class:`~textattack.shared.AttackChecpoint`]`):
                Path of saved checkpoint or the actual saved checkpoint.
        """
        assert isinstance(
            checkpoint, (str, textattack.shared.AttackCheckpoint)
        ), f"`checkpoint` must be of type `str` or `textattack.shared.AttackCheckpoint`, but got type `{type(checkpoint)}`."

        if isinstance(checkpoint, str):
            checkpoint = textattack.shared.AttackCheckpoint.load(checkpoint)
        attacker = cls(attack, dataset, checkpoint.attack_args)
        attacker.attack_log_manager = checkpoint.attack_log_manager
        attacker._checkpoint = checkpoint
        return attacker

    @staticmethod
    def attack_interactive(attack):
        print(attack, "\n")

        print("Running in interactive mode")
        print("----------------------------")

        while True:
            print('Enter a sentence to attack or "q" to quit:')
            text = input()

            if text == "q":
                break

            if not text:
                continue

            print("Attacking...")

            example = textattack.shared.attacked_text.AttackedText(text)
            output = attack.goal_function.get_output(example)
            result = attack.attack(example, output)
            print(result.__str__(color_method="ansi") + "\n")


#
# Helper Methods for multiprocess attacks
#
def pytorch_multiprocessing_workaround():
    # This is a fix for a known bug
    try:
        torch.multiprocessing.set_start_method("spawn", force=True)
        torch.multiprocessing.set_sharing_strategy("file_system")
    except RuntimeError:
        pass


def set_env_variables(gpu_id):
    # Disable tensorflow logs, except in the case of an error.
    if "TF_CPP_MIN_LOG_LEVEL" not in os.environ:
        os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

    # Set sharing strategy to file_system to avoid file descriptor leaks
    torch.multiprocessing.set_sharing_strategy("file_system")

    # Only use one GPU, if we have one.
    # For Tensorflow
    # TODO: Using USE with `--parallel` raises similar issue as https://github.com/tensorflow/tensorflow/issues/38518#
    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    # For PyTorch
    torch.cuda.set_device(gpu_id)

    # Fix TensorFlow GPU memory growth
    try:
        import tensorflow as tf

        gpus = tf.config.experimental.list_physical_devices("GPU")
        if gpus:
            try:
                # Currently, memory growth needs to be the same across GPUs
                gpu = gpus[gpu_id]
                tf.config.experimental.set_visible_devices(gpu, "GPU")
                tf.config.experimental.set_memory_growth(gpu, True)
            except RuntimeError as e:
                print(e)
    except ModuleNotFoundError:
        pass


def attack_from_queue(
    attack, attack_args, num_gpus, first_to_start, lock, in_queue, out_queue
):
    assert isinstance(
        attack, Attack
    ), f"`attack` must be of type `Attack`, but got type `{type(attack)}`."

    gpu_id = (torch.multiprocessing.current_process()._identity[0] - 1) % num_gpus
    set_env_variables(gpu_id)
    textattack.shared.utils.set_seed(attack_args.random_seed)
    if torch.multiprocessing.current_process()._identity[0] > 1:
        logging.disable()

    attack.cuda_()

    # Simple non-synchronized check to see if it's the first process to reach this point.
    # This let us avoid waiting for lock.
    if bool(first_to_start.value):
        # If it's first process to reach this step, we first try to acquire the lock to update the value.
        with lock:
            # Because another process could have changed `first_to_start=False` while we wait, we check again.
            if bool(first_to_start.value):
                first_to_start.value = 0
                if not attack_args.silent:
                    print(attack, "\n")

    while True:
        try:
            i, example, ground_truth_output = in_queue.get(timeout=5)
            if i == "END" and example == "END" and ground_truth_output == "END":
                # End process when sentinel value is received
                break
            else:
                result = attack.attack(example, ground_truth_output)
                out_queue.put((i, result))
        except Exception as e:
            if isinstance(e, queue.Empty):
                continue
            else:
                out_queue.put((i, (e, traceback.format_exc())))


In [15]:
# @title
"""Adversarial Attacks in a Hard Label Black Box Setting
====================================

Reimplementation of search method from Generating Natural Language Attack in a Hard Label Black Box Setting by Maheshwary et.al

`https://arxiv.org/abs/2012.14956`_

`<https://github.com/RishabhMaheshwary/hard-label-attack>`_
"""

from collections import defaultdict
import logging
import numpy as np
from scipy.special import softmax
import torch

from textattack.constraints.semantics.sentence_encoders import UniversalSentenceEncoder
from textattack.goal_function_results import GoalFunctionResultStatus
from textattack.search_methods import PopulationBasedSearch, PopulationMember
from textattack.shared.validators import transformation_consists_of_word_swaps


class HardLabelAttack(PopulationBasedSearch):

    """Attacks a model with word substitutions by observing only the topmost
    predicted label. It consist of three key steps: (1) Random Initialization
    (2) Search Space Reduction and (3) Genetic Based Optimization. We use the
    hyper-parameters values as mentioned in the original paper. Following the
    original implementation of the paper we set the window size used for
    measuring semantic similarity as 40 and the trials used for generating a
    random adversarial example to 2500. The trials can be reduced to speed up
    attack but it will lower down the  success rate of the attack.

    Args:
        pop_size (:obj:`int`, optional): The population size. Defaults to 30.
        max_iters (:obj:`int`, optional): The maximum number of iterations to use. Defaults to 100.
        max_replacements_per_index (:obj:`int`, optional): Maximum number of mutations that can be sone on an index. Defaults to 25.
        window_size (:obj:`int`, optional): The window of text around the changed index to be considered for measuring semantic similarity. Defaults to 40.
        trials (:obj:`int`, optional): The number of trials for generating random adversarial example. Defaults to 2500.
    """

    def __init__(
        self,
        pop_size=30,
        max_iters=100,
        max_replacements_per_index=25,
        window_size=40,
        trials=2500,
    ):
        self.max_iters = max_iters
        self.pop_size = pop_size
        self.trials = trials
        self.max_replacements_per_index = max_replacements_per_index

        self._search_over = False
        self.window_size = window_size
        self.sim_metric = torch.nn.CosineSimilarity(dim=1)
        self.use = UniversalSentenceEncoder()

    def _get_similarity_score(
        self, transformed_texts, initial_text, window_around_idx=-1, use_window=False
    ):
        """
        Calculates semantic similarity betweem the `transformed_texts` and `initial_text`
        Args:
            transformed_texts (list): The perturbed or mutated text inputs.
            initial_text (AttackedText): The original text input.
            window_around_idx (int): The index around which the window to consider.
            use_window (bool): Calculate similarity with or without window.
        Returns:
            scores as `list[float]`
        """

        if use_window:
            texts_within_window = []
            for txt in transformed_texts:
                texts_within_window.append(
                    txt.text_window_around_index(window_around_idx, self.window_size)
                )
            transformed_texts = texts_within_window
            initial_text = initial_text.text_window_around_index(
                window_around_idx, self.window_size
            )

        embeddings_transformed_texts = self.use.encode(transformed_texts)
        embeddings_initial_text = self.use.encode(
            [initial_text] * len(transformed_texts)
        )

        if not isinstance(embeddings_transformed_texts, torch.Tensor):
            embeddings_transformed_texts = torch.tensor(embeddings_transformed_texts)

        if not isinstance(embeddings_initial_text, torch.Tensor):
            embeddings_initial_text = torch.tensor(embeddings_initial_text)

        scores = self.sim_metric(embeddings_transformed_texts, embeddings_initial_text)
        return scores

    def _perturb(
        self, pop_member, original_result, initial_text, index_to_perturb, best_attack
    ):
        """
        Mutates or perturbes a population member `pop_member` at `index_to_perturb`
        Args:
            pop_member (PopulationMember): The pop_member to perturb or mutate.
            original_result (GoalFunctionResult): The result of `pop_member`.
            initial_text (AttackedText): The original text input.
            index_to_perturb (int): The index to be perturbed or mutated.
            best_attack (PopulationMember): The best attack until now.
        Returns:
            pop_member as `PopulationMember`
        """

        cur_adv_text = pop_member.attacked_text
        # First try to replace the substituted word back with the original word.
        perturbed_text = cur_adv_text.replace_word_at_index(
            index_to_perturb, initial_text.words[index_to_perturb]
        )
        results, search_over = self.get_goal_results([perturbed_text])
        if search_over:
            return PopulationMember(
                cur_adv_text,
                attributes={
                    "similarity_score": -1,
                    "valid_adversarial_example": False,
                },
            )
        if results[0].goal_status == GoalFunctionResultStatus.SUCCEEDED:
            similarity_score = self._get_similarity_score(
                [perturbed_text.text], initial_text.text
            )[0]
            return PopulationMember(
                perturbed_text,
                attributes={
                    "similarity_score": similarity_score,
                    "valid_adversarial_example": True,
                },
            )
        # Replace with other synonyms.
        perturbed_texts = self.get_transformations(
            perturbed_text,
            original_text=initial_text,
            indices_to_modify=[index_to_perturb],
        )

        final_perturbed_texts = []

        for txt in perturbed_texts:

            passed_constraints = self._check_constraints(
                txt, perturbed_text, original_text=initial_text
            )

            if (
                passed_constraints
                and txt.words[index_to_perturb] != cur_adv_text.words[index_to_perturb]
            ):
                final_perturbed_texts.append(txt)

        results, _ = self.get_goal_results(final_perturbed_texts)

        perturbed_adv_texts = []
        perturbed_adv_strings = []
        for i in range(len(results)):

            if results[i].goal_status == GoalFunctionResultStatus.SUCCEEDED:
                perturbed_adv_texts.append(final_perturbed_texts[i])
                perturbed_adv_strings.append(final_perturbed_texts[i].text)

        if len(perturbed_adv_texts) == 0:
            return PopulationMember(
                cur_adv_text,
                attributes={
                    "similarity_score": pop_member.attributes["similarity_score"],
                    "valid_adversarial_example": False,
                },
            )

        prev_similarity_scores = self._get_similarity_score(
            [pop_member.attacked_text], initial_text, index_to_perturb, use_window=True
        )
        similarity_scores = self._get_similarity_score(
            perturbed_adv_texts, initial_text, index_to_perturb, use_window=True
        )

        best_similarity_score = -1
        for i in range(len(similarity_scores)):
            # Filter out candidates which do not improve the semantic similarity
            # score within `self.window_size`.
            if similarity_scores[i] < prev_similarity_scores[0]:
                continue

            if similarity_scores[i] > best_similarity_score:
                best_similarity_score = similarity_scores[i]
                final_adv_text = perturbed_adv_texts[i]

        if best_similarity_score == -1:
            return PopulationMember(
                cur_adv_text,
                attributes={
                    "similarity_score": best_similarity_score,
                    "valid_adversarial_example": False,
                },
            )

        new_pop_member = PopulationMember(
            final_adv_text,
            attributes={
                "similarity_score": best_similarity_score,
                "valid_adversarial_example": True,
            },
        )
        return new_pop_member

    def _initialize_population(self, cur_adv_text, initial_text, changed_indices):
        """
        Initialize a population of size `pop_size` by mutating the indices in `changed_indices`
        Args:
            cur_adv_text (AttackedText): The adversarial text obtained after search space reduction step.
            initial_text (AttackedText): The original input text.
            changed_indices (list): The indices changed in the `cur_adv_text` from the `initial_text`.
        Returns:
            population as `list[PopulationMember]`
        """
        results, _ = self.get_goal_results([cur_adv_text])
        similarity_score = self._get_similarity_score(
            [cur_adv_text.text], initial_text.text
        )[0]

        initial_pop_member = PopulationMember(
            cur_adv_text,
            attributes={
                "similarity_score": similarity_score,
                "valid_adversarial_example": True,
            },
        )
        population = []
        # Randomly select `self.pop_size - 1` substituted indices for mutation.
        indices = np.random.choice(len(changed_indices), size=self.pop_size - 1)
        for i in range(len(indices)):

            new_pop_member = self._perturb(
                initial_pop_member,
                results[0],
                initial_text,
                changed_indices[indices[i]],
                initial_pop_member,
            )

            if new_pop_member.attributes["valid_adversarial_example"]:
                population.append(new_pop_member)

        return population

    def _generate_initial_adversarial_example(self, initial_result):
        """Generates a random initial adversarial example.

        Args:
            initial_result (GoalFunctionResult): Original result.
        Returns:
            initial_adversarial_text as `AttackedText`
        """
        initial_text = initial_result.attacked_text
        len_text = len(initial_text.words)

        for trial in range(self.trials):

            random_index_order = np.arange(len_text)
            np.random.shuffle(random_index_order)
            random_text = initial_text
            random_generated_texts = []
            for idx in random_index_order:

                transformed_texts = self.get_transformations(
                    random_text, original_text=initial_text, indices_to_modify=[idx]
                )

                if len(transformed_texts) > 0:
                    random_text = np.random.choice(transformed_texts)
                    random_generated_texts.append(random_text)
            results, _ = self.get_goal_results(random_generated_texts)

            for i in range(len(results)):
                if results[i].goal_status == GoalFunctionResultStatus.SUCCEEDED:
                    return results[i].attacked_text, results[i]
        return initial_text, initial_result

    def _search_space_reduction(self, initial_result, initial_adv_text):
        """Reduces the count of perturbed words in the `initial_adv_text`.

        Args:
            initial_result (GoalFunctionResult): Original result.
            initial_adv_text (int): The adversarial text obtained after random initialization.
        Returns:
            initial_adv_text as `AttackedText`
        """
        original_text = initial_result.attacked_text
        different_indices = original_text.all_words_diff(initial_adv_text)
        replacements = []
        txts = []
        # Replace the substituted words with their original counterparts.
        for idx in different_indices:

            new_text = initial_adv_text.replace_word_at_index(
                idx, initial_result.attacked_text.words[idx]
            )
            # Filter out replacements that are not adversarial.
            results, search_over = self.get_goal_results([new_text])
            if search_over:
                return initial_adv_text
            if results[0].goal_status == GoalFunctionResultStatus.SUCCEEDED:
                txts.append(new_text.text)
                replacements.append((idx, initial_result.attacked_text.words[idx]))

        if len(txts) == 0:
            return initial_adv_text

        similarity_scores = self._get_similarity_score(
            txts, initial_result.attacked_text.text
        )
        # Sort the replacements based on semantic similarity and replace
        # the substitute words with their original counterparts till the
        # remains adversarial.
        replacements = list(zip(similarity_scores, replacements))
        replacements.sort()
        replacements.reverse()

        for score, replacement in replacements:

            new_text = initial_adv_text.replace_word_at_index(
                replacement[0], replacement[1]
            )
            results, _ = self.get_goal_results([new_text])

            if results[0].goal_status == GoalFunctionResultStatus.SUCCEEDED:
                initial_adv_text = new_text
            else:
                break
        return initial_adv_text

    def _crossover(self, text_1, text_2, initial_text, best_attack):
        """Does crossover step on `text_1` and `text_2` to yield a child text.

        Args:
            text_1 (AttackedText): Parent text 1.
            text_2 (AttackedText): Parent text 2.
            initial_text (AttackedText): The original text input.
            best_attack (PopulationMember): The best adversarial attack until now.
        Returns:
            pop_member as `PopulationMember`
        """
        index_to_replace = []
        words_to_replace = []

        for i in range(len(text_1.words)):

            if np.random.uniform() < 0.5:
                index_to_replace.append(i)
                words_to_replace.append(text_2.words[i])

        new_text = text_1.attacked_text.replace_words_at_indices(
            index_to_replace, words_to_replace
        )
        num_child_different_words = len(new_text.all_words_diff(initial_text))
        num_best_different_words = len(
            best_attack.attacked_text.all_words_diff(initial_text)
        )
        # If the generated child has more perturbation, return.
        if num_child_different_words > num_best_different_words:
            return PopulationMember(
                text_1,
                attributes={
                    "similarity_score": best_attack.attributes["similarity_score"],
                    "valid_adversarial_example": False,
                },
            )
        else:
            for idx in index_to_replace:
                if new_text.words[idx] == text_1.words[idx]:
                    continue
                previous_similarity_score = self._get_similarity_score(
                    [best_attack.attacked_text],
                    initial_text,
                    window_around_idx=idx,
                    use_window=True,
                )
                new_similarity_score = self._get_similarity_score(
                    [new_text], initial_text, window_around_idx=idx, use_window=True
                )
                # filter out child texts which do not improve semantic similarity
                # within a `self.window_size`.
                if new_similarity_score < previous_similarity_score:
                    return PopulationMember(
                        text_1,
                        attributes={
                            "similarity_score": new_similarity_score,
                            "valid_adversarial_example": False,
                        },
                    )

            similarity_score = self._get_similarity_score(
                [new_text.text], initial_text.text
            )
            return PopulationMember(
                new_text,
                attributes={
                    "similarity_score": similarity_score,
                    "valid_adversarial_example": True,
                },
            )

    def perform_search(self, initial_result):
        """Performs random Initialization, Search Space Reduction, and genetic
        Optimization to generate an attack by observing only the topmost label.

        Args:
            initial_result (GoalFunctionResult): The result of the original input text.
        Returns:
            best_result as GoalFunctionResult
        """
        # Random Initialization
        initial_adv_text, init_adv_result = self._generate_initial_adversarial_example(initial_result)

        if initial_result.attacked_text.words == initial_adv_text.words:
            return initial_result

        # Search space reduction
        new_text = self._search_space_reduction(initial_result, initial_adv_text)
        results, search_over = self.get_goal_results([new_text])

        if search_over:
            return init_adv_result

        if results[0].goal_status != GoalFunctionResultStatus.SUCCEEDED:
            return results[0]

        initial_text = initial_result.attacked_text
        changed_indices = initial_text.all_words_diff(new_text)
        changed_indices = list(changed_indices)
        self._search_over = False

        # Initialize Population
        population = self._initialize_population(
            new_text, initial_text, changed_indices
        )
        if len(population) == 0:
            return results[0]

        similarity_score = self._get_similarity_score(
            [initial_adv_text.text], initial_text.text
        )[0]

        best_attack = PopulationMember(
            new_text,
            attributes={
                "similarity_score": similarity_score,
                "valid_adversarial_example": True,
            },
        )
        best_result, _ = self.get_goal_results([best_attack.attacked_text])
        #print('best_result',best_result)

        max_replacements_left = defaultdict(int)

        for _ in range(self.max_iters):

            num_min_different_words = len(
                best_attack.attacked_text.all_words_diff(initial_text)
            )

            similarity_scores = []

            for pop_member in population:
                num_different_words = len(
                    pop_member.attacked_text.all_words_diff(initial_text)
                )

                # Extract similarity score and ensure it's a scalar value (float/int)
                score = pop_member.attributes["similarity_score"]

                if isinstance(score, (list, np.ndarray)):
                    # If the score is a list or array, take its first element
                    score = score[0] if isinstance(score, list) else score.item()

                similarity_scores.append(score)

                if num_different_words < num_min_different_words:
                    best_attack = pop_member
                    num_min_different_words = num_different_words


            results, _ = self.get_goal_results([best_attack.attacked_text])

            if not results:
                logging.warning("get_goal_results returned an empty list for the attacked text.")
                return initial_result  # Fallback to the initial result or handle appropriately

            best_result = results[0]

            if best_result.goal_status != GoalFunctionResultStatus.SUCCEEDED:
                logging.error("Goal status is not SUCCEEDED for the best result.")
                return best_result  # Or handle the error as needed

            # Proceed with the rest of the code

            assert best_result.goal_status == GoalFunctionResultStatus.SUCCEEDED

            # Validate the goal status
            assert best_result.goal_status == GoalFunctionResultStatus.SUCCEEDED

            # Ensure the goal status is as expected
            assert best_result.goal_status == GoalFunctionResultStatus.SUCCEEDED

            similarity_scores = [score.item() if isinstance(score, torch.Tensor) else score for score in similarity_scores]
            # Now similarity_scores will be a list of scalars: [0.9421, 0.9157, 0.9086]
            similarity_scores = np.array(similarity_scores)
            # Ensure similarity_scores is a 1D array of scalars
            similarity_scores = np.array(similarity_scores).flatten()  # Flatten to 1D
            similarity_scores = softmax(similarity_scores)  # Apply softmax

            # Ensure we have valid values before passing to np.random.choice
            assert np.all(np.isfinite(similarity_scores)), "Similarity scores contain invalid values!"

            # Selection
            parent1_idx = np.random.choice(len(population), size=self.pop_size - 1, p=similarity_scores)
            parent2_idx = np.random.choice(len(population), size=self.pop_size - 1, p=similarity_scores)

            childs = []
            child_texts = []
            final_childs = []

            # Crossover
            for i in range(self.pop_size - 1):

                new_child = self._crossover(
                    population[parent1_idx[i]],
                    population[parent2_idx[i]],
                    initial_text,
                    best_attack,
                )
                if new_child.attributes["valid_adversarial_example"]:
                    childs.append(new_child)
                    child_texts.append(new_child.attacked_text)

            results, _ = self.get_goal_results(child_texts)

            for i in range(len(results)):
                if results[i].goal_status == GoalFunctionResultStatus.SUCCEEDED:
                    final_childs.append((childs[i], results[i]))

            # In the original implementation of the paper the changed indices are not
            # updated in every iteration. Here, the changed indices are updated.
            changed_indices = initial_text.all_words_diff(best_attack.attacked_text)
            changed_indices = list(changed_indices)
            index_positions_to_perturb = np.random.choice(
                len(changed_indices), size=min(len(changed_indices), len(final_childs))
            )

            population = []

            # Mutation of children
            for i in range(len(index_positions_to_perturb)):
                child = final_childs[i][0]
                j = index_positions_to_perturb[i]

                if (
                    initial_text.words[changed_indices[j]]
                    == child.attacked_text.words[changed_indices[j]]
                ):
                    population.append(child)
                    continue

                if (
                    max_replacements_left[changed_indices[j]]
                    <= self.max_replacements_per_index
                ):
                    new_pop_member = self._perturb(
                        child,
                        final_childs[i][1],
                        initial_text,
                        changed_indices[j],
                        best_attack,
                    )
                    if new_pop_member.attributes["valid_adversarial_example"]:
                        population.append(new_pop_member)
                        max_replacements_left[changed_indices[j]] += 1

            population.append(best_attack)

        return best_result



    def check_transformation_compatibility(self, transformation):
        """The genetic algorithm is specifically designed for word
        substitutions."""
        return transformation_consists_of_word_swaps(transformation)

    @property
    def is_black_box(self):
        return True

    def extra_repr_keys(self):
        return ["pop_size", "max_iters", "max_replacements_per_index"]



## Classe do Ataque GNLA

In [16]:
# @title
"""
Hard Label Black Box Attack
==================================
(Generating Natural Language Attack in a Hard Label Black Box Setting)
"""
from textattack.constraints.grammaticality import PartOfSpeech
from textattack.constraints.pre_transformation import (
    InputColumnModification,
    StopwordModification,
)
from textattack.constraints.semantics import WordEmbeddingDistance
from textattack.goal_functions import UntargetedClassification
#from textattack import Attack
from textattack.transformations import WordSwapEmbedding

"""
Attack Recipe Class
========================

"""

from abc import ABC, abstractmethod

#from textattack import Attack


class AttackRecipe(Attack, ABC):
    """A recipe for building an NLP adversarial attack from the literature."""

    @staticmethod
    @abstractmethod
    def build(model_wrapper, **kwargs):
        """Creates pre-built :class:`~textattack.Attack` that correspond to
        attacks from the literature.

        Args:
            model_wrapper (:class:`~textattack.models.wrappers.ModelWrapper`):
                :class:`~textattack.models.wrappers.ModelWrapper` that contains the victim model and tokenizer.
                This is passed to :class:`~textattack.goal_functions.GoalFunction` when constructing the attack.
            kwargs:
                Additional keyword arguments.
        Returns:
            :class:`~textattack.Attack`
        """
        raise NotImplementedError()



class HardLabelMaheshwary2021(AttackRecipe):
    """Rishabh Maheshwary, Saket Maheshwary, Vikram Pudi (2021).

    Generating Natural Language Attack in a Hard Label Black Box
    Setting. https://arxiv.org/abs/2012.14956 Methodology description
    quoted from the paper: We propose a decision-based attack strategy
    that crafts high quality adversarial examples on text classification
    and entailment tasks. Our proposed attack strategy leverages
    population-based optimization algorithm to craft plausible and
    semantically similar adversarial examples by observing only the top
    label predicted by the target model. At each iteration, the
    optimization procedure allow word replacements that maximizes the
    overall semantic similarity between the original and the adversarial
    text.
    """

    @staticmethod
    def build(model, pop_size=30, max_iters=100, min_cos_sim=0.5):
        transformation = WordSwapEmbedding(max_candidates=50)
        #
        # Don't modify the stopwords defined
        # in the paper public implementation.
        #
        stopwords = set(
            [
                "a",
                "about",
                "above",
                "across",
                "after",
                "afterwards",
                "again",
                "against",
                "ain",
                "all",
                "almost",
                "alone",
                "along",
                "already",
                "also",
                "although",
                "am",
                "among",
                "amongst",
                "an",
                "and",
                "another",
                "any",
                "anyhow",
                "anyone",
                "anything",
                "anyway",
                "anywhere",
                "are",
                "aren",
                "aren't",
                "around",
                "as",
                "at",
                "back",
                "been",
                "before",
                "beforehand",
                "behind",
                "being",
                "below",
                "beside",
                "besides",
                "between",
                "beyond",
                "both",
                "but",
                "by",
                "can",
                "cannot",
                "could",
                "couldn",
                "couldn't",
                "d",
                "didn",
                "didn't",
                "doesn",
                "doesn't",
                "don",
                "don't",
                "down",
                "due",
                "during",
                "either",
                "else",
                "elsewhere",
                "empty",
                "enough",
                "even",
                "ever",
                "everyone",
                "everything",
                "everywhere",
                "except",
                "first",
                "for",
                "former",
                "formerly",
                "from",
                "hadn",
                "hadn't",
                "hasn",
                "hasn't",
                "haven",
                "haven't",
                "he",
                "hence",
                "her",
                "here",
                "hereafter",
                "hereby",
                "herein",
                "hereupon",
                "hers",
                "herself",
                "him",
                "himself",
                "his",
                "how",
                "however",
                "hundred",
                "i",
                "if",
                "in",
                "indeed",
                "into",
                "is",
                "isn",
                "isn't",
                "it",
                "it's",
                "its",
                "itself",
                "just",
                "latter",
                "latterly",
                "least",
                "ll",
                "may",
                "me",
                "meanwhile",
                "mightn",
                "mightn't",
                "mine",
                "more",
                "moreover",
                "most",
                "mostly",
                "must",
                "mustn",
                "mustn't",
                "my",
                "myself",
                "namely",
                "needn",
                "needn't",
                "neither",
                "never",
                "nevertheless",
                "next",
                "no",
                "nobody",
                "none",
                "noone",
                "nor",
                "not",
                "nothing",
                "now",
                "nowhere",
                "o",
                "of",
                "off",
                "on",
                "once",
                "one",
                "only",
                "onto",
                "or",
                "other",
                "others",
                "otherwise",
                "our",
                "ours",
                "ourselves",
                "out",
                "over",
                "per",
                "please",
                "s",
                "same",
                "shan",
                "shan't",
                "she",
                "she's",
                "should've",
                "shouldn",
                "shouldn't",
                "somehow",
                "something",
                "sometime",
                "somewhere",
                "such",
                "t",
                "than",
                "that",
                "that'll",
                "the",
                "their",
                "theirs",
                "them",
                "themselves",
                "then",
                "thence",
                "there",
                "thereafter",
                "thereby",
                "therefore",
                "therein",
                "thereupon",
                "these",
                "they",
                "this",
                "those",
                "through",
                "throughout",
                "thru",
                "thus",
                "to",
                "too",
                "toward",
                "towards",
                "under",
                "unless",
                "until",
                "up",
                "upon",
                "used",
                "USER"
                "ve",
                "was",
                "wasn",
                "wasn't",
                "we",
                "were",
                "weren",
                "weren't",
                "what",
                "whatever",
                "when",
                "whence",
                "whenever",
                "where",
                "whereafter",
                "whereas",
                "whereby",
                "wherein",
                "whereupon",
                "wherever",
                "whether",
                "which",
                "while",
                "whither",
                "who",
                "whoever",
                "whole",
                "whom",
                "whose",
                "why",
                "with",
                "within",
                "without",
                "won",
                "won't",
                "would",
                "wouldn",
                "wouldn't",
                "y",
                "yet",
                "you",
                "you'd",
                "you'll",
                "you're",
                "you've",
                "your",
                "yours",
                "yourself",
                "yourselves",
            ]
        )
        # fmt: on
        constraints = [StopwordModification(stopwords=stopwords)]
        #
        # During entailment, we should only edit the hypothesis - keep the premise
        # the same.
        #
        input_column_modification = InputColumnModification(
            ["premise", "hypothesis"], {"premise"}
        )
        constraints.append(input_column_modification)
        #
        # Minimum word embedding cosine similarity of 0.5.
        #
        constraints.append(WordEmbeddingDistance(min_cos_sim=0.5))
        #
        # Only replace words with the same part of speech (or nouns with verbs)
        #
        constraints.append(PartOfSpeech(allow_verb_noun_swap=True))
        #
        # Goal is untargeted classification
        #
        goal_function = UntargetedClassification(model)
        #
        # Generate attack using only the topmost predicted label.
        #
        search_method = HardLabelAttack(pop_size=30, max_iters=100)

        return Attack(goal_function, constraints, transformation, search_method)


# Realizando Ataque

## Atividade: Variação Paramétrica do Ataque GNLA

1.  - num_examples: 4
    - pop_size: 30
    - max_iters: 100
    - min_cos_sim: 0.5
    - Resultado:
        
        
        | Attack Results                |           |
        |-------------------------------|-----------|
        | Number of successful attacks: | 1         |
        | Number of failed attacks:     | 1         |
        | Number of skipped attacks:    | 2         |
        | Original accuracy:            | 50.0%     |
        | Accuracy under attack:        | 25.0%     |
        | Attack success rate:          | 50.0%     |
        | Average perturbed word %:     | 20.0%     |
        | Average num. words per input: | 8.5       |
        | Avg num queries:              | 6517.5    |
        | Average Original Perplexity:  | 130691.62 |
        | Average Attack Perplexity:    | 16750.64  |
        | Average Attack USE Score:     | 0.83      |


2.  - num_examples: 10
    - pop_size: 10
    - max_iters: 150
    - min_cos_sim: 0.6
    - Resultado:
        
        | Attack Results                |         |
        |-------------------------------|---------|
        | Number of successful attacks: | 2       |
        | Number of failed attacks:     | 3       |
        | Number of skipped attacks:    | 5       |
        | Original accuracy:            | 50.0%   |
        | Accuracy under attack:        | 30.0%   |
        | Attack success rate:          | 40.0%   |
        | Average perturbed word %:     | 22.5%   |
        | Average num. words per input: | 8.1     |
        | Avg num queries:              | 6320.2  |
        | Average Original Perplexity:  | 9837.1  |
        | Average Attack Perplexity:    | 5079.99 |
        | Average Attack USE Score:     | 0.77    |

3.  - num_examples: 15
    - pop_size: 10
    - max_iters: 125
    - min_cos_sim: 0.55
    - Resultado:
        
        
        | Attack Results                |         |
        |-------------------------------|---------|
        | Number of successful attacks: | 5       |
        | Number of failed attacks:     | 5       |
        | Number of skipped attacks:    | 5       |
        | Original accuracy:            | 66.67%  |
        | Accuracy under attack:        | 33.33%  |
        | Attack success rate:          | 50.0%   |
        | Average perturbed word %:     | 14.95%  |
        | Average num. words per input: | 8.73    |
        | Avg num queries:              | 5441.4  |
        | Average Original Perplexity:  | 1865.72 |
        | Average Attack Perplexity:    | 1843.19 |
        | Average Attack USE Score:     | 0.85    |


4.  - num_examples: 15
    - pop_size: 7
    - max_iters: 200
    - min_cos_sim: 0.55
    - Resultado:
        
        | Attack Results                |         |
        |-------------------------------|---------|
        | Number of successful attacks: | 5       |
        | Number of failed attacks:     | 5       |
        | Number of skipped attacks:    | 5       |
        | Original accuracy:            | 66.67%  |
        | Accuracy under attack:        | 33.33%  |
        | Attack success rate:          | 50.0%   |
        | Average perturbed word %:     | 14.95%  |
        | Average num. words per input: | 8.73    |
        | Avg num queries:              | 5454.8  |
        | Average Original Perplexity:  | 1865.72 |
        | Average Attack Perplexity:    | 2070.42 |
        | Average Attack USE Score:     | 0.84    |
        
        `original_text`  \
        3   someone vetaken piece [[shit]] volcano   
        6   oh noes tough [[shit]]   
        11  difference kavanaugh one men admitted [[groping]] 15 year old girl years ago going confirmed scj demsarefrauds demsaredone walkawaydemocrats 2018 redwave voteredsaveamerica trumptrain maga	   
        12  lying corrupt [[traitor]] nobody wants hear anymore lies deepstatecorruption	
        13  [[scare]] every american playing hockey warped puck	

        `perturbed_text`  
        3   someone vetaken piece [[bordello]] volcano  
        6   oh noes tough [[christ]]  
        11  difference kavanaugh one men admitted [[masturbate]] 15 year old girl years ago going confirmed scj demsarefrauds demsaredone walkawaydemocrats 2018 redwave voteredsaveamerica trumptrain maga  
        12  lying corrupt [[turncoat]] nobody wants hear anymore lies deepstatecorruption 
        13  [[shitless]] every american playing hockey warped puck   

In [34]:
detoxify_model = DetoxifyVictimModel()
attack = HardLabelMaheshwary2021.build(
    model=detoxify_model,
    pop_size=7,
    max_iters=200,
    min_cos_sim=0.55
    )

textattack: Unknown if model of class <class 'detoxify.detoxify.Detoxify'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.


In [35]:
attack_args = AttackArgs(
    num_examples=15,  # Specify the number of examples you want to attack
    log_to_csv="gnla_attack_detoxify",  # Log results to a CSV
    disable_stdout=False,
    parallel=False, 
    enable_advance_metrics=True,
    query_budget=7000 # o artigo indica uma media de 7k para um ataque bem sucedido
)


attacker = Attacker(attack, dataset, attack_args)

attacker.attack_dataset()

textattack: Logging to CSV at path gnla_attack_detoxify/2026-11-12-19-28-log.csv


Attack(
  (search_method): HardLabelAttack(
    (pop_size):  30
    (max_iters):  100
    (max_replacements_per_index):  25
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapEmbedding(
    (max_candidates):  50
    (embedding):  WordEmbedding
  )
  (constraints): 
    (0): WordEmbeddingDistance(
        (embedding):  WordEmbedding
        (min_cos_sim):  0.5
        (cased):  False
        (include_unknown_words):  True
        (compare_against_original):  True
      )
    (1): PartOfSpeech(
        (tagger_type):  nltk
        (tagset):  universal
        (allow_verb_noun_swap):  True
        (compare_against_original):  True
      )
    (2): StopwordModification
    (3): InputColumnModification(
        (matching_column_labels):  ['premise', 'hypothesis']
        (columns_to_ignore):  {'premise'}
      )
  (is_black_box):  True
) 



--------------------------------------------- Result 1 ---------------------------------------------
[[0 (95%)]] --> [[[SKIPPED]]]

ask native americans take




--------------------------------------------- Result 2 ---------------------------------------------
[[0 (88%)]] --> [[[SKIPPED]]]

go home drunk maga trump 2020




--------------------------------------------- Result 3 ---------------------------------------------
[[0 (100%)]] --> [[[FAILED]]]

amazon investigating chinese employees selling internal data third party sellers looking edge competitive marketplace amazon maga kag china tcot




--------------------------------------------- Result 4 ---------------------------------------------
[[1 (98%)]] --> [[0 (100%)]]

someone vetaken piece [[shit]] volcano

someone vetaken piece [[bordello]] volcano




... (5 outputs left out)

--------------------------------------------- Result 13 ---------------------------------------------
[[1 (98%)]] --> [[0 (74%)]]

lying corrupt [[traitor]] nobody wants hear anymore lies deepstatecorruption

lying corrupt [[turncoat]] nobody wants hear anymore lies deepstatecorruption




--------------------------------------------- Result 14 ---------------------------------------------
[[0 (81%)]] --> [[1 (93%)]]

[[scare]] every american playing hockey warped puck

[[shitless]] every american playing hockey warped puck




[Succeeded / Failed / Skipped / Total] 5 / 5 / 5 / 15: 100%|██████████| 15/15 [28:58<00:00, 115.87s/it]


--------------------------------------------- Result 15 ---------------------------------------------
[[0 (100%)]] --> [[[FAILED]]]

like soda like like boarders lot ice



+-------------------------------+---------+
| Attack Results                |         |
+-------------------------------+---------+
| Number of successful attacks: | 5       |
| Number of failed attacks:     | 5       |
| Number of skipped attacks:    | 5       |
| Original accuracy:            | 66.67%  |
| Accuracy under attack:        | 33.33%  |
| Attack success rate:          | 50.0%   |
| Average perturbed word %:     | 14.95%  |
| Average num. words per input: | 8.73    |
| Avg num queries:              | 5454.8  |
| Average Original Perplexity:  | 1865.72 |
| Average Attack Perplexity:    | 2070.42 |
| Average Attack USE Score:     | 0.84    |
+-------------------------------+---------+


In [36]:
import glob
import os
csv_files = glob.glob("/content/gnla_attack_detoxify/*.csv")
latest_file = max(csv_files, key=os.path.getctime)
print("Arquivo encontrado:", latest_file)

Arquivo encontrado: /content/gnla_attack_detoxify/2026-11-12-19-28-log.csv


In [37]:
df = pd.read_csv(latest_file)

df.head()

,original_text,perturbed_text,original_score,perturbed_score,original_output,perturbed_output,ground_truth_output,num_queries,result_type
0,ask native americans take,ask native americans take,0.945867,0.945867,0,0,1,1,Skipped
1,go home drunk maga trump 2020,go home drunk maga trump 2020,0.882088,0.882088,0,0,1,1,Skipped
2,amazon investigating chinese employees selling...,amazon investigating chinese employees selling...,0.001254,0.001254,0,0,0,7000,Failed
3,someone vetaken piece [[shit]] volcano,someone vetaken piece [[bordello]] volcano,0.020911,0.997101,1,0,1,6035,Successful
4,obama wanted liberals illegals move red states,obama wanted liberals illegals move red states,0.036349,0.036349,0,0,0,7000,Failed


In [38]:
df.loc[df.result_type=='Successful'][['original_text', 'perturbed_text']]


,original_text,perturbed_text
3,someone vetaken piece [[shit]] volcano,someone vetaken piece [[bordello]] volcano
6,oh noes tough [[shit]],oh noes tough [[christ]]
11,difference kavanaugh one men admitted [[gropin...,difference kavanaugh one men admitted [[mastur...
12,lying corrupt [[traitor]] nobody wants hear an...,lying corrupt [[turncoat]] nobody wants hear a...
13,[[scare]] every american playing hockey warped...,[[shitless]] every american playing hockey war...
